# 调用模型  

### 阻塞式调用

In [13]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

# 获取环境变量中的API密钥和基础URL
api_key = os.getenv("DASHSCOPE_API_KEY")
base_url = os.getenv("DASHSCOPE_BASE_URL")

model = init_chat_model(
    model="deepseek-v4-flash", #模型名称
    #model ="qwen3.5-flash", #模型切换为qwen3.5
    model_provider="openai", #因为不支持百炼,所以使用兼容的openai接口
    temperature=0.1,
    api_key=api_key,
    base_url=base_url
)

print(type(model))

response = model.invoke("你是谁")
print(response.content) 

<class 'langchain_openai.chat_models.base.ChatOpenAI'>
你好！我是DeepSeek，由深度求索公司创造的AI助手。我是一个纯文本模型，可以帮你回答问题、处理信息、进行对话等。我的一些特点包括：

- **免费使用**：完全免费，无需付费
- **超长上下文**：支持1M上下文，可以一次性处理像《三体》三部曲那样的大部头书籍
- **文件处理**：支持上传图像、txt、pdf、ppt、word、excel等文件，并从中读取文字信息
- **联网搜索**：支持联网搜索功能（需要手动开启）
- **语音输入**：App端支持语音输入

我的知识截止于2025年5月，会尽力为你提供准确、有用的帮助。有什么我可以帮你的吗？😊


### 流式调用

In [15]:
#流式调用
response = model.stream("你是谁")
for chunk in response:
    print(chunk.content, end="", flush=True)

你好！我是DeepSeek，由深度求索公司创造的AI助手。😊

我是一个纯文本模型，知识截止日期为2025年5月。我支持：
- 📝 阅读链接和处理上传的文件（图像、txt、pdf、ppt、word、ppt、word、excel等）
- 🌐 联网搜索（需要手动开启）
- 💬 超长上下文（1M tokens，可以一次性处理《三体》三部曲这样的长文）
- 🆓 完全免费使用

我的特点包括：
1. **强大的推理能力**：在数学、编程等领域表现出色
2. **多语言支持**：能用你使用的语言回复
3. **文件处理**：从上传的文件中提取文字信息
4. **回复风格**：热情、细腻，注重细节

有什么我可以帮你的吗？无论是回答问题、写作、编程还是其他需求，我都很乐意协助你！✨

### 在Agent中调用模型  

langchain提供一个create_agent()方法，该方法会根据指定的模型和工具列表创建一个Agent对象。创建时需要指定一个模型:
- 使用初始化好的模型对象
- 使用模型名称,让langchain自动选择一个模型

In [ ]:
from langchain.agents import create_agent
#使用初始化好的模型,初始化代码在上面init_chat_model那里
agent = create_agent(model=model)
#使用模型名称创建模型,langchain会自动根据模型名称选择api与key
#agent = create_agent(model="qwen3.5-flash")


调用支持两种方式:
- 流式调用
- 阻塞式调用

#### 阻塞式调用

In [26]:
response = agent.invoke({
    "messages": [
        {"role": "user", "content": "你是谁?"}
    ]
}
)

print(response)

{'messages': [HumanMessage(content='你是谁?', additional_kwargs={}, response_metadata={}, id='a79fff83-9869-4a1f-a9f6-717b8761e52d'), AIMessage(content='你好！我是DeepSeek，由深度求索公司创造的AI助手。😊\n\n我是一个纯文本模型，擅长回答各种问题、进行对话交流、帮助处理文档等。虽然我不支持多模态识别（比如直接识别图片内容），但我可以：\n\n- ✅ 阅读链接内容\n- ✅ 处理上传的文件（图像、txt、pdf、ppt、word、excel等），从中提取文字信息\n- ✅ 支持联网搜索（需要手动开启）\n- ✅ 拥有1M的超长上下文，可以一次性处理像《三体》三部曲那么大体量的内容\n\n而且最重要的是——**我完全免费**！无论是网页版和App都可以使用！App还支持语音输入功能。\n\n我的知识截止日期是2025年5月，会尽我所能为你提供准确、有用的帮助。有什么我可以帮你的吗？😊', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 299, 'prompt_tokens': 6, 'total_tokens': 305, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 120, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': None, 'id': 'chatcmpl-fe14b982-c841-9497-94b2-bf41f4c9ea13', 'finish_reason': '

#### 流式调用

In [29]:
messages = agent.stream(
    {
        "messages": [
            {"role": "system", "content": "你是以为渗透测试专家."},
            {"role": "user", "content": "你可以做什么?"},
        ]
    },
    stream_mode="messages"

)
print(type(messages))
print(messages)

for token, metada in messages:
    if token.content:
        print(token.content, end="", flush=True)


<class 'generator'>
<generator object Pregel.stream at 0x3a365fc0>
作为渗透测试专家，我可以帮助你完成以下任务：

1. **漏洞分析与利用**：识别Web应用、网络设备、操作系统中的安全漏洞（如SQL注入、XSS、CSRF、命令注入等），并提供利用方法或修复建议。

2. **渗透测试流程指导**：从信息收集、漏洞扫描、权限提升到横向移动，提供完整的测试策略和工具使用建议（如Nmap、Burp Suite、Metasploit等）。

3. **安全加固建议**：根据发现的漏洞，给出代码修复、配置优化、网络隔离等防御措施。

4. **红队/蓝队模拟**：模拟攻击者行为（如钓鱼攻击、社会工程学），或协助防御方设计检测规则。

5. **工具与脚本定制**：编写或优化渗透测试脚本（Python、Bash等），自动化漏洞验证或数据提取。

6. **报告与文档**：生成符合行业标准的渗透测试报告，包括风险等级、复现步骤、修复方案。

**注意**：所有操作需在合法授权范围内进行，我仅提供技术指导，不参与任何未授权活动。** 如果你有具体场景（如测试某个Web应用或网络环境），可以进一步描述，我会给出针对性方案。